<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration — Algorithm</b></h1>
</div>


## Purpose

This notebook is the **complete implementation specification** for the corresponding `main.ipynb`.

Its purpose is to make the executable notebook fully reproducible from the algorithm alone. A competent reader should be able to rebuild `main.ipynb` **without guessing, inventing, or choosing any meaningful implementation detail**.

The specification therefore defines the exact execution order, libraries, constants, data structures, helper procedures, matrix operations, numerical branches, failure conditions, validation tolerances, diagnostic figures, output filenames, and final checks used by the implementation.

**Rule:** if a coding decision is required to reproduce `main.ipynb`, that decision must be stated explicitly in this notebook.

# Complete Algorithm Specification

The pseudocode below is the authoritative coding blueprint for `main.ipynb`. It is intentionally explicit: every computational step, stored variable, helper function, branch, numerical operation, diagnostic figure, output filename, and validation rule required to reproduce the executable notebook is stated directly.

```text
ALGORITHM Camera_Calibration

  # 0. IMPORTS, CONSTANTS, PATHS, AND REQUIRED OUTPUTS
  IMPORT Path FROM pathlib
  IMPORT cv2
  IMPORT numpy AS np
  IMPORT matplotlib.pyplot AS plt

  CALL np.set_printoptions(precision=6, suppress=True)

  SQUARE_SIZE_M ← 0.03
  INTERNAL_CORNERS_X ← 8
  INTERNAL_CORNERS_Y ← 6
  MIN_VALID_VIEWS ← 3

  OUTPUT_DIR ← Path("../outputs/figures")
  CALL OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

  DATA_DIR ← Path("../data/calibration_images")
  IMAGE_EXTENSIONS ← {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

  REFINE_CRITERIA ← (
      cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
      30,
      0.001
  )

  REQUIRED_OUTPUTS ← [
      "detected_chessboard_corners.png",
      "homography_estimation_pipeline.png",
      "estimated_camera_poses.png",
      "reprojection_results.png",
      "mean_reprojection_error_by_view.png",
      "reprojection_error_distribution.png"
  ]

  # 1. LOAD THE SORTED CALIBRATION IMAGES
  IF NOT DATA_DIR.is_dir():
      RAISE FileNotFoundError(
          f"Calibration image directory not found: {DATA_DIR}"
      )

  calibration_image_paths ← sorted([
      path
      FOR path IN DATA_DIR.iterdir()
      IF path.is_file()
      AND path.suffix.lower() IN IMAGE_EXTENSIONS
  ])

  IF NOT calibration_image_paths:
      RAISE FileNotFoundError(
          f"No supported calibration images found in: {DATA_DIR}"
      )

  readable_calibration_views ← []
  views ← []

  FOR image_path IN calibration_image_paths:
      image ← cv2.imread(str(image_path))

      IF image IS None:
          PRINT f"[SKIPPED] {image_path.name}: image could not be read"
          CONTINUE

      readable_calibration_views.append({
          "path": image_path,
          "name": image_path.name,
          "image": image
      })

  IF NOT readable_calibration_views:
      RAISE RuntimeError("No readable calibration images are available.")

  PRINT f"Calibration files found : {len(calibration_image_paths)}"
  PRINT f"Readable views          : {len(readable_calibration_views)}"

  # 2. DETECT AND REFINE CHESSBOARD CORNERS
  FUNCTION detect_and_refine_chessboard_corners(
      image,
      corners_x=INTERNAL_CORNERS_X,
      corners_y=INTERNAL_CORNERS_Y
  ):
      gray ← cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

      found, corners ← cv2.findChessboardCorners(
          gray,
          (corners_x, corners_y),
          None
      )

      IF (NOT found) OR (corners IS None):
          RETURN None

      refined ← cv2.cornerSubPix(
          gray,
          corners,
          (11, 11),
          (-1, -1),
          REFINE_CRITERIA
      )

      RETURN refined.reshape(-1, 2)

  views ← []

  FOR readable_calibration_view IN readable_calibration_views:
      detected_corner_points ← detect_and_refine_chessboard_corners(
          readable_calibration_view["image"]
      )

      IF detected_corner_points IS None:
          PRINT f"[SKIPPED] {readable_calibration_view['name']}: chessboard not detected"
          CONTINUE

      view ← readable_calibration_view.copy()
      view["im_pts"] ← detected_corner_points
      views.append(view)

      PRINT f"[OK] {view['name']}: {detected_corner_points.shape[0]} refined corners"

  IF len(views) < MIN_VALID_VIEWS:
      RAISE RuntimeError(
          f"At least {MIN_VALID_VIEWS} valid calibration views are required; "
          f"only {len(views)} were detected."
      )

  PRINT f"\nValid calibration views: {len(views)}"

  # 3. BUILD THE PLANAR WORLD COORDINATES
  FUNCTION build_chessboard_plane_points(
      corners_x=INTERNAL_CORNERS_X,
      corners_y=INTERNAL_CORNERS_Y,
      square_size=SQUARE_SIZE_M
  ):
      points ← np.zeros((corners_x * corners_y, 2), dtype=np.float64)
      index ← 0

      FOR y_index IN range(corners_y):
          FOR x_index IN range(corners_x):
              points[index, 0] ← x_index * square_size
              points[index, 1] ← y_index * square_size
              index ← index + 1

      RETURN points

  chessboard_plane_points ← build_chessboard_plane_points()

  FOR view IN views:
      view["plane_pts"] ← chessboard_plane_points.copy()

  PRINT f"Planar points per view: {chessboard_plane_points.shape[0]}"
  PRINT f"Square size: {SQUARE_SIZE_M:.3f} m"

  # 4. COMPUTE T_image AND T_plane
  FUNCTION convert_to_homogeneous_coordinates(points):
      points ← np.asarray(points, dtype=float)
      homogeneous ← np.ones((points.shape[0], 3), dtype=float)
      homogeneous[:, :2] ← points
      RETURN homogeneous

  FUNCTION compute_point_normalization_transform(points):
      points ← np.asarray(points, dtype=float)
      centroid ← np.mean(points, axis=0)
      shifted ← points - centroid
      distances ← np.linalg.norm(shifted, axis=1)
      mean_distance ← np.mean(distances)

      IF mean_distance < 1e-12:
          RAISE ValueError(
              "Point normalization is undefined for coincident points."
          )

      scale ← np.sqrt(2.0) / mean_distance

      RETURN np.array([
          [scale, 0.0, -scale * centroid[0]],
          [0.0, scale, -scale * centroid[1]],
          [0.0, 0.0, 1.0]
      ], dtype=float)

  FOR view IN views:
      T_image ← compute_point_normalization_transform(view["im_pts"])
      T_plane ← compute_point_normalization_transform(view["plane_pts"])

      view["T_image"] ← T_image
      view["T_plane"] ← T_plane

      view["normalized_image"] ← (
          T_image
          @ convert_to_homogeneous_coordinates(view["im_pts"]).T
      ).T

      view["normalized_plane"] ← (
          T_plane
          @ convert_to_homogeneous_coordinates(view["plane_pts"]).T
      ).T

  PRINT "Image and planar point normalization completed."

  # 5. BUILD Q AND SOLVE Qh = 0 BY SVD
  FUNCTION build_homography_dlt_matrix(normalized_plane, normalized_image):
      n_points ← normalized_image.shape[0]
      Q ← np.zeros((2 * n_points, 9), dtype=float)

      FOR i IN range(n_points):
          X, Y, _ ← normalized_plane[i]
          u, v, _ ← normalized_image[i]

          Q[2 * i] ← [
              X, Y, 1.0, 0.0, 0.0, 0.0,
              -u * X, -u * Y, -u
          ]

          Q[2 * i + 1] ← [
              0.0, 0.0, 0.0, X, Y, 1.0,
              -v * X, -v * Y, -v
          ]

      RETURN Q

  FOR view IN views:
      Q ← build_homography_dlt_matrix(
          view["normalized_plane"],
          view["normalized_image"]
      )

      _, singular_values_Q, Vt_Q ← np.linalg.svd(Q)
      h ← Vt_Q[-1]
      H_normalized ← h.reshape(3, 3)

      view["Q"] ← Q
      view["h"] ← h
      view["dlt_singular_values"] ← singular_values_Q
      view["H_normalized"] ← H_normalized

  PRINT "Normalized DLT solved for all views."

  # 6. DENORMALIZE EACH HOMOGRAPHY
  FUNCTION denormalize_homography_matrix(H_normalized, T_image, T_plane):
      H ← np.linalg.inv(T_image) @ H_normalized @ T_plane

      IF abs(H[2, 2]) < 1e-12:
          RAISE ValueError(
              "Degenerate homography: H[2, 2] is too close to zero."
          )

      H ← H / H[2, 2]
      RETURN H

  expected_corner_count ← INTERNAL_CORNERS_X * INTERNAL_CORNERS_Y

  FOR view IN views:
      view["H"] ← denormalize_homography_matrix(
          view["H_normalized"],
          view["T_image"],
          view["T_plane"]
      )

      IF view["im_pts"].shape != (expected_corner_count, 2):
          RAISE ValueError(f"Unexpected image-point shape for {view['name']}.")

      IF view["plane_pts"].shape != (expected_corner_count, 2):
          RAISE ValueError(f"Unexpected planar-point shape for {view['name']}.")

      IF NOT np.all(np.isfinite(view["H"])):
          RAISE ValueError(f"Non-finite homography for {view['name']}.")

  PRINT "All homographies denormalized and validated."

  # 7. BUILD THE ZHANG MATRIX V AND SOLVE Vb = 0 BY SVD
  FUNCTION build_zhang_constraint_vector(hi, hj):
      RETURN np.array([
          hi[0] * hj[0],
          hi[0] * hj[1] + hi[1] * hj[0],
          hi[1] * hj[1],
          hi[2] * hj[0] + hi[0] * hj[2],
          hi[2] * hj[1] + hi[1] * hj[2],
          hi[2] * hj[2]
      ], dtype=float)

  FUNCTION build_zhang_constraints_for_view(H):
      h1 ← H[:, 0]
      h2 ← H[:, 1]
      v12 ← build_zhang_constraint_vector(h1, h2)
      v11 ← build_zhang_constraint_vector(h1, h1)
      v22 ← build_zhang_constraint_vector(h2, h2)
      RETURN np.vstack((v12, v11 - v22))

  V ← np.vstack([
      build_zhang_constraints_for_view(view["H"])
      FOR view IN views
  ])

  _, singular_values, Vt ← np.linalg.svd(V)
  b ← Vt[-1]

  PRINT "Constraint matrix shape:", V.shape
  PRINT "\nSingular values:"
  PRINT singular_values
  PRINT "\nResidual norm ||Vb||:"
  PRINT np.linalg.norm(V @ b)
  PRINT "\nEstimated b vector:"
  PRINT b

  # 8. RECOVER alpha, beta, gamma, u0, v0 AND CONSTRUCT K
  b11, b12, b22, b13, b23, b33 ← b
  denominator ← b11 * b22 - b12**2

  IF abs(denominator) < 1e-12:
      RAISE ValueError("Degenerate intrinsic calibration constraints.")

  v0 ← (b12 * b13 - b11 * b23) / denominator
  lambda_ ← b33 - (
      b13**2
      + v0 * (b12 * b13 - b11 * b23)
  ) / b11

  IF (lambda_ / b11 <= 0) OR (lambda_ * b11 / denominator <= 0):
      b ← -b
      b11, b12, b22, b13, b23, b33 ← b
      denominator ← b11 * b22 - b12**2
      v0 ← (b12 * b13 - b11 * b23) / denominator
      lambda_ ← b33 - (
          b13**2
          + v0 * (b12 * b13 - b11 * b23)
      ) / b11

  alpha ← np.sqrt(lambda_ / b11)
  beta ← np.sqrt(lambda_ * b11 / denominator)
  gamma ← -b12 * alpha**2 * beta / lambda_
  u0 ← gamma * v0 / beta - b13 * alpha**2 / lambda_

  K ← np.array([
      [alpha, gamma, u0],
      [0.0, beta, v0],
      [0.0, 0.0, 1.0]
  ], dtype=float)

  PRINT "Intrinsic parameters"
  PRINT "--------------------"
  PRINT f"alpha : {alpha:.6f}"
  PRINT f"beta  : {beta:.6f}"
  PRINT f"gamma : {gamma:.6f}"
  PRINT f"u0    : {u0:.6f}"
  PRINT f"v0    : {v0:.6f}"
  PRINT "\nIntrinsic matrix K:"
  PRINT K

  # 9. RECOVER R AND t FOR EVERY RETAINED VIEW
  FUNCTION recover_camera_pose(K, H):
      K_inv ← np.linalg.inv(K)
      h1 ← H[:, 0]
      h2 ← H[:, 1]
      h3 ← H[:, 2]

      lambda_p ← 1.0 / np.linalg.norm(K_inv @ h1)

      r1 ← lambda_p * (K_inv @ h1)
      r2 ← lambda_p * (K_inv @ h2)
      r3 ← np.cross(r1, r2)

      R_approx ← np.column_stack((r1, r2, r3))
      U, _, Vt ← np.linalg.svd(R_approx)
      R ← U @ Vt

      IF np.linalg.det(R) < 0:
          U[:, -1] ← -U[:, -1]
          R ← U @ Vt

      t ← lambda_p * (K_inv @ h3)
      RETURN R, t

  FOR view IN views:
      R, t ← recover_camera_pose(K, view["H"])
      view["R"] ← R
      view["t"] ← t

      PRINT view["name"]
      PRINT "R ="
      PRINT R
      PRINT "t =", t
      PRINT "-" * 60

  # 10. REPROJECT THE Z = 0 CALIBRATION POINTS
  FOR view IN views:
      chessboard_xy_points ← view["plane_pts"]

      chessboard_xyz_points ← np.column_stack((
          chessboard_xy_points,
          np.zeros(chessboard_xy_points.shape[0])
      ))

      camera_frame_points ← (
          chessboard_xyz_points @ view["R"].T
          + view["t"]
      )

      projected_h ← camera_frame_points @ K.T

      IF np.any(np.abs(projected_h[:, 2]) < 1e-12):
          RAISE ValueError(
              f"Invalid homogeneous depth during reprojection for {view['name']}."
          )

      reprojected_image_points ← (
          projected_h[:, :2]
          / projected_h[:, 2:3]
      )

      view["world_xyz"] ← chessboard_xyz_points
      view["projected"] ← reprojected_image_points

  PRINT f"Reprojected calibration views: {len(views)}"

  # 11. COMPUTE POINT-WISE ERRORS, MEAN ERROR, AND RMSE
  FOR view IN views:
      residuals ← view["projected"] - view["im_pts"]
      errors ← np.linalg.norm(residuals, axis=1)

      view["residuals"] ← residuals
      view["errors"] ← errors
      view["mean_error"] ← float(np.mean(errors))
      view["rmse"] ← float(np.sqrt(np.mean(errors**2)))

      PRINT view["name"]
      PRINT f"Mean reprojection error : {view['mean_error']:.4f} px"
      PRINT f"Reprojection RMSE       : {view['rmse']:.4f} px"
      PRINT "-" * 60

  all_reprojection_errors ← np.concatenate([
      view["errors"]
      FOR view IN views
  ])

  overall_mean_error ← float(np.mean(all_reprojection_errors))
  overall_rmse ← float(np.sqrt(np.mean(all_reprojection_errors**2)))

  PRINT "Calibration summary"
  PRINT "-------------------"
  PRINT f"Valid calibration views : {len(views)}"
  PRINT f"Calibration points       : {len(all_reprojection_errors)}"
  PRINT f"Mean reprojection error  : {overall_mean_error:.4f} px"
  PRINT f"Overall RMSE             : {overall_rmse:.4f} px"

  # 12. PRODUCE AND SAVE THE SIX REQUIRED DIAGNOSTIC FIGURES
  PRINT "Diagnostic figures to generate:"
  FOR output_name IN REQUIRED_OUTPUTS:
      PRINT f"  - {output_name}"

  # 12.1 Homography Estimation Pipeline
  fig, ax ← plt.subplots(figsize=(15, 3.5))
  CALL ax.axis("off")

  labels ← [
      "World points\n(X, Y, 1)",
      "Image points\n(u, v, 1)",
      "Normalize\nT_plane, T_image",
      "Build Q\nQh = 0",
      "SVD\nSolve for h",
      "Denormalize\nH"
  ]

  x_positions ← np.linspace(0.08, 0.92, len(labels))

  FOR x, label IN zip(x_positions, labels):
      CALL ax.text(
          x, 0.5, label,
          ha="center", va="center", fontsize=12,
          transform=ax.transAxes,
          bbox={
              "boxstyle": "round,pad=0.5",
              "fill": False,
              "linewidth": 1.5
          }
      )

  FOR start, end IN zip(x_positions[:-1], x_positions[1:]):
      CALL ax.annotate(
          "",
          xy=(end - 0.055, 0.5),
          xytext=(start + 0.055, 0.5),
          xycoords=ax.transAxes,
          arrowprops={
              "arrowstyle": "->",
              "linewidth": 1.5
          }
      )

  CALL ax.set_title("Homography Estimation Pipeline", fontsize=16, pad=20)
  CALL plt.tight_layout()
  CALL fig.savefig(
      OUTPUT_DIR / "homography_estimation_pipeline.png",
      dpi=300, bbox_inches="tight"
  )
  CALL plt.show()

  # 12.2 Mean Reprojection Error by View
  image_names ← [view["name"] FOR view IN views]
  mean_reprojection_errors ← [view["mean_error"] FOR view IN views]

  fig, ax ← plt.subplots(figsize=(10, 5))
  bars ← ax.bar(image_names, mean_reprojection_errors)

  CALL ax.axhline(
      overall_mean_error,
      linestyle="--",
      linewidth=1.5,
      label=f"Overall mean = {overall_mean_error:.3f} px"
  )

  FOR bar, value IN zip(bars, mean_reprojection_errors):
      CALL ax.text(
          bar.get_x() + bar.get_width() / 2,
          bar.get_height(),
          f"{value:.2f}",
          ha="center", va="bottom", fontsize=9
      )

  CALL ax.set_title("Mean Reprojection Error by Calibration View")
  CALL ax.set_xlabel("Calibration image")
  CALL ax.set_ylabel("Mean reprojection error (pixels)")
  CALL ax.grid(axis="y", alpha=0.25)
  CALL ax.legend()
  CALL plt.tight_layout()
  CALL fig.savefig(
      OUTPUT_DIR / "mean_reprojection_error_by_view.png",
      dpi=300, bbox_inches="tight"
  )
  CALL plt.show()

  # 12.3 Reprojection Error Distribution
  fig, ax ← plt.subplots(figsize=(9, 5))
  CALL ax.hist(
      all_reprojection_errors,
      bins=20,
      edgecolor="black",
      alpha=0.8
  )

  CALL ax.axvline(
      overall_mean_error,
      linestyle="--",
      linewidth=1.5,
      label=f"Mean = {overall_mean_error:.3f} px"
  )

  CALL ax.axvline(
      overall_rmse,
      linestyle=":",
      linewidth=1.5,
      label=f"RMSE = {overall_rmse:.3f} px"
  )

  CALL ax.set_title("Distribution of Reprojection Errors")
  CALL ax.set_xlabel("Reprojection error (pixels)")
  CALL ax.set_ylabel("Number of calibration points")
  CALL ax.grid(axis="y", alpha=0.25)
  CALL ax.legend()
  CALL plt.tight_layout()
  CALL fig.savefig(
      OUTPUT_DIR / "reprojection_error_distribution.png",
      dpi=300, bbox_inches="tight"
  )
  CALL plt.show()

  # 12.4 Detected Chessboard Corners
  number_of_views ← len(views)
  plot_columns ← 3
  plot_rows ← int(np.ceil(number_of_views / plot_columns))

  fig, axes ← plt.subplots(
      plot_rows,
      plot_columns,
      figsize=(15, 5 * plot_rows)
  )

  axes ← np.asarray(axes).reshape(-1)

  FOR ax, view IN zip(axes, views):
      visual ← view["image"].copy()

      corners ← (
          view["im_pts"]
          .reshape(-1, 1, 2)
          .astype(np.float32)
      )

      CALL cv2.drawChessboardCorners(
          visual,
          (INTERNAL_CORNERS_X, INTERNAL_CORNERS_Y),
          corners,
          True
      )

      visual_rgb ← cv2.cvtColor(visual, cv2.COLOR_BGR2RGB)
      CALL ax.imshow(visual_rgb)
      CALL ax.set_title(view["name"])
      CALL ax.axis("off")

  FOR ax IN axes[number_of_views:]:
      CALL ax.axis("off")

  CALL plt.tight_layout()
  CALL fig.savefig(
      OUTPUT_DIR / "detected_chessboard_corners.png",
      dpi=300, bbox_inches="tight"
  )
  CALL plt.show()

  # 12.5 Estimated Camera Poses
  fig ← plt.figure(figsize=(10, 8))
  ax ← fig.add_subplot(111, projection="3d")

  reference_chessboard_xy_points ← views[0]["plane_pts"]
  reference_chessboard_xyz_points ← np.column_stack((
      reference_chessboard_xy_points,
      np.zeros(reference_chessboard_xy_points.shape[0])
  ))

  CALL ax.scatter(
      reference_chessboard_xyz_points[:, 0],
      reference_chessboard_xyz_points[:, 1],
      reference_chessboard_xyz_points[:, 2],
      marker="s",
      label="Chessboard"
  )

  FOR view IN views:
      R ← view["R"]
      t ← view["t"]
      camera_center ← -R.T @ t

      CALL ax.scatter(
          camera_center[0],
          camera_center[1],
          camera_center[2],
          marker="o"
      )

      CALL ax.text(
          camera_center[0],
          camera_center[1],
          camera_center[2],
          view["path"].stem,
          fontsize=8
      )

  CALL ax.set_title("Estimated Camera Poses")
  CALL ax.set_xlabel("X (m)")
  CALL ax.set_ylabel("Y (m)")
  CALL ax.set_zlabel("Z (m)")
  CALL ax.legend()
  CALL plt.tight_layout()
  CALL fig.savefig(
      OUTPUT_DIR / "estimated_camera_poses.png",
      dpi=300, bbox_inches="tight"
  )
  CALL plt.show()

  # 12.6 Detected vs Reprojected Points
  number_of_views ← len(views)
  plot_columns ← 3
  plot_rows ← int(np.ceil(number_of_views / plot_columns))

  fig, axes ← plt.subplots(
      plot_rows,
      plot_columns,
      figsize=(15, 5 * plot_rows)
  )

  axes ← np.asarray(axes).reshape(-1)

  FOR ax, view IN zip(axes, views):
      image_rgb ← cv2.cvtColor(
          view["image"],
          cv2.COLOR_BGR2RGB
      )

      CALL ax.imshow(image_rgb)

      CALL ax.scatter(
          view["im_pts"][:, 0],
          view["im_pts"][:, 1],
          s=24,
          marker="o",
          label="Detected"
      )

      CALL ax.scatter(
          view["projected"][:, 0],
          view["projected"][:, 1],
          s=24,
          marker="x",
          label="Reprojected"
      )

      CALL ax.set_title(
          f"{view['name']} — RMSE: {view['rmse']:.3f} px"
      )
      CALL ax.axis("off")
      CALL ax.legend(fontsize=8)

  FOR ax IN axes[number_of_views:]:
      CALL ax.axis("off")

  CALL plt.tight_layout()
  CALL fig.savefig(
      OUTPUT_DIR / "reprojection_results.png",
      dpi=300, bbox_inches="tight"
  )
  CALL plt.show()

  # 13. RUN THE NUMERICAL AND OUTPUT-FILE VALIDATION CHECKS
  IF K.shape != (3, 3) OR NOT np.all(np.isfinite(K)):
      RAISE ValueError("Invalid intrinsic matrix K.")

  FOR view IN views:
      IF NOT np.all(np.isfinite(view["H"])):
          RAISE ValueError(f"Non-finite homography for {view['name']}.")

      R ← view["R"]

      IF NOT np.allclose(
          R.T @ R,
          np.eye(3),
          atol=1e-6
      ):
          RAISE ValueError(f"Non-orthonormal rotation for {view['name']}.")

      IF NOT np.isclose(
          np.linalg.det(R),
          1.0,
          atol=1e-6
      ):
          RAISE ValueError(f"Invalid rotation determinant for {view['name']}.")

      IF NOT np.all(np.isfinite(view["errors"])):
          RAISE ValueError(f"Non-finite residuals for {view['name']}.")

  missing ← [
      name
      FOR name IN REQUIRED_OUTPUTS
      IF NOT (OUTPUT_DIR / name).exists()
  ]

  IF missing:
      RAISE FileNotFoundError(
          "Missing outputs: " + ", ".join(missing)
      )

  PRINT "All Camera Calibration validation checks passed."

END ALGORITHM
```
